In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [49]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

In [52]:
train_df = pd.read_csv("Training.csv")

# Remove unwanted columns
train_df = train_df.loc[:, ~train_df.columns.str.contains('^Unnamed')]

print("Training dataset shape:", train_df.shape)

Training dataset shape: (4920, 133)


In [54]:
train_df.head()

,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
3,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
4,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection


In [53]:
test_df = pd.read_csv("Testing.csv")

test_df = test_df.loc[:, ~test_df.columns.str.contains('^Unnamed')]

print("Testing dataset shape:", test_df.shape)

Testing dataset shape: (42, 133)


In [55]:
test_df.head()

,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
1,0,0,0,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Allergy
2,0,0,0,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,GERD
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Chronic cholestasis
4,1,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,Drug Reaction


In [56]:
X_train = train_df.drop("prognosis", axis=1)
y_train = train_df["prognosis"]

In [57]:
X_test = test_df.drop("prognosis", axis=1)
y_test = test_df["prognosis"]

In [58]:
le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [59]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss"
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [61]:
pred = model.predict(X_test)

In [62]:
accuracy = accuracy_score(y_test, pred)

print("Test Accuracy:", accuracy)

Test Accuracy: 0.9761904761904762


In [63]:
print(classification_report(
    y_test,
    pred,
    target_names=le.classes_
))

                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00         1
                                   AIDS       1.00      1.00      1.00         1
                                   Acne       1.00      1.00      1.00         1
                    Alcoholic hepatitis       1.00      1.00      1.00         1
                                Allergy       1.00      1.00      1.00         1
                              Arthritis       1.00      1.00      1.00         1
                       Bronchial Asthma       1.00      1.00      1.00         1
                   Cervical spondylosis       1.00      1.00      1.00         1
                            Chicken pox       0.50      1.00      0.67         1
                    Chronic cholestasis       1.00      1.00      1.00         1
                            Common Cold       1.00      1.00      1.00         1
                           

In [64]:
def predict_top3(symptom_vector):

    probs = model.predict_proba([symptom_vector])[0]

    top3 = np.argsort(probs)[-3:][::-1]

    print("Top Predictions:\n")

    for rank, i in enumerate(top3, 1):
        disease = le.inverse_transform([i])[0]
        confidence = probs[i] * 100

        print(f"{rank}. {disease} ({confidence:.2f}%)")

In [65]:
# Create a patient symptom vector
sample = np.zeros(X.shape[1])

# Symptoms present (set to 1)
symptoms = [
    "fever",
    "headache",
    "vomiting",
    "fatigue",
    "chills"
]

# Convert symptoms to vector
for symptom in symptoms:
    if symptom in X.columns:
        sample[X.columns.get_loc(symptom)] = 1

# Predict top 3 diseases
predict_top3(sample)

Top Predictions:

1. Paralysis (brain hemorrhage) (24.12%)
2. Allergy (4.67%)
3. Typhoid (2.69%)


In [51]:
df.duplicated().sum()

np.int64(4616)

In [66]:
import joblib
joblib.dump(model, "disease_prediction_model.pkl")

['disease_prediction_model.pkl']

In [67]:
joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']

In [71]:
!ls /content

disease_prediction_model.pkl  sample_data  Training.csv
label_encoder.pkl	      Testing.csv
